# Notebook 09: defences and what they cost

**Class 03_03.** Three defences against the attacks of notebooks 07 and 08, each measured on clean data (does it
still detect?) and under attack (does it resist?):

* **A. Ensembles:** several detectors that see the data differently; an attacker must fool all of them.
* **B. Robust features:** score only the features the attack cannot change without changing what it does.
* **C. Vetted retraining:** retrain only on data that a frozen, trusted model also accepts.

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl

sys.path.insert(0, ".")
import plots_robust as pr

import anomlib as an
import deeplib as dl
import robustlib as rl

an.table_config()
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
mut = np.array([n in rl.MUTABLE for n in sp.prep.names])
Xatt, cat = Xte[y == 1], sp.test["category"].to_numpy()[y == 1]


def assess(name: str, score_tr, X_va, X_te, X_att, X_att_mimic) -> dict:
    """Clean metrics at 1% FPR and the mimicry evasion rate among the attacks caught."""
    s_va = score_tr(X_va)
    t = an.threshold(s_va)
    caught = score_tr(X_att) > t
    passed = score_tr(X_att_mimic[caught]) <= t
    return {
        "detector": name,
        **an.evaluate(score_tr(X_te), y, s_va),
        "caught": caught.mean(),
        "mimicry evasion": passed.mean(),
    }


Xmim = rl.mimicry(Xatt, Xtr, mut)

## A. Ensembles

* **Rank mean:** each member's score becomes its rank among normal validation scores (a number in [0, 1]); the
  ensemble score is the mean rank.
* **OR rule:** alarm if *any* member alarms; each member gets 1/m of the false-alarm budget (threshold at the
  $1-0.01/m$ quantile), so the ensemble still makes about 1% false alarms.

In [ ]:
members = [an.Mahalanobis(), an.GMM(), an.KNNDist(), an.PCARecon(), dl.DeepSVDD()]
for m in members:
    m.fit(Xtr)
rows = [assess(m.name, m.score, Xva, Xte, Xatt, Xmim) for m in members]
ens = rl.RankEnsemble(members, Xva)
ens.ref = [np.sort(m.score(Xva)) for m in members]  # members are already fitted
rows.append(assess("rank-mean ensemble", ens.score, Xva, Xte, Xatt, Xmim))
q = 1 - 0.01 / len(members)
ts = [np.quantile(m.score(Xva), q) for m in members]


def or_score(X):
    """max over members of score / own threshold: > 1 means at least one member alarms."""
    return np.max([m.score(X) / t for m, t in zip(members, ts)], axis=0)


rows.append(assess("OR ensemble", or_score, Xva, Xte, Xatt, Xmim))
print(pl.DataFrame(rows))
pr.ensembles(rows)  # slide plots: assets/figures/plots/03/

The rank-mean ensemble is the best clean detector of the module without any tuning (AUPRC 0.69, AUROC 0.99): its
members make different mistakes. Against mimicry it is *between* its members (0.065), not better than the most
robust one (Mahalanobis, 0.018): averaging ranks lets a large drop in one member's score pull the mean below the
threshold. The OR rule is stricter (0.043) but pays in false alarms when members disagree. An ensemble raises the
cost of attacks *tuned to one model*; it does not make weak features strong.

## B. Robust features

Fit the same detectors on the **fixed** features only (64 of 71): mimicry of volume and timing then cannot change
the score at all. The price is what the dropped features contributed on clean data.

In [ ]:
fx = ~mut
rows = []
for make in [an.Mahalanobis, an.GMM, an.PCARecon]:
    full, fixed = make().fit(Xtr), make().fit(Xtr[:, fx])
    rows.append(assess(full.name + " (all features)", full.score, Xva, Xte, Xatt, Xmim))
    rows.append(assess(fixed.name + " (fixed features)", fixed.score, Xva[:, fx], Xte[:, fx], Xatt[:, fx], Xmim[:, fx]))
print(pl.DataFrame(rows))
pr.robust_features(rows)

By construction mimicry evasion drops to 0. The clean price depends on the detector: PCA loses almost nothing
(recall 0.78 to 0.75), the Mahalanobis distance loses a quarter of its recall (0.68 to 0.52), because volume and
timing did carry part of the evidence. Choose the features by what an attacker can change, then measure the price.

## C. Vetted retraining against the boiling frog

Notebook 08's attack, but each round the defender keeps only the new points that a **frozen reference** model
(fitted once on trusted data) also accepts at a looser threshold (0.1% false alarms). Legitimate slow change still
gets in; the poison has to stay inside the *original* normal region, so it cannot walk to the target.

In [ ]:
X0 = an.example1(500, seed=0)
target = 2 * an.EX1_POINTS["B"]
ref = an.Mahalanobis().fit(X0)
t_ref = an.threshold(ref.score(X0), fpr=0.001)


def frog(vet: bool) -> tuple[list[float], int]:
    """The boiling frog of notebook 08 on Example 1; with `vet` the defender keeps only the new points that the
    frozen reference model also accepts. Returns score(target) / threshold per round and the poison points kept."""
    rng = np.random.default_rng(0)
    data, ratios, kept = X0.copy(), [], 0
    for r in range(21):
        det = an.Mahalanobis().fit(data)
        t = an.threshold(det.score(data))
        ratios.append(float(det.score(target[None])[0] / t))
        c = data.mean(0)
        line = c + np.linspace(0, 1, 200)[:, None] * (target - c)
        ok = line[det.score(line) < 0.98 * t]
        poison = (ok[-1] if len(ok) else c) + rng.normal(0, 0.05, (25, 2))
        poison = poison[det.score(poison) < t]
        if vet:
            poison = poison[ref.score(poison) < t_ref]  # the vetting step
        kept += len(poison)
        data = np.vstack([data, poison])[-500:]
    return ratios, kept


ratios, kept = frog(vet=True)
plain, _ = frog(vet=False)
print("score(target)/threshold every 4 rounds:", [round(x, 2) for x in ratios[::4]])
print(f"poison points that passed the vetting: {kept} of {25 * 20}; target accepted: {min(ratios) < 1}")
print("without vetting, every 4 rounds:", [round(x, 2) for x in plain[::4]])
pr.vetting(plain, ratios)

The vetting lets in 56 of the 500 poison points (those inside the frozen normal region); the model moves a little
(the ratio falls from 8.5 to 4.5) and then stops: the target is never accepted.

## D. Summary: every defence has a price and an answer

| defence | stops | attacker's answer | cost |
|:--|:--|:--|:--|
| ensemble (OR) | attacks tuned to one detector | fool every member (more queries) | more compute; false alarms add up |
| robust features | cheap mimicry of volume and timing | change the attack itself (slower, noisier) | clean recall on attacks that differed in volume |
| vetted retraining | boiling frog | stay inside the frozen region (slow, small) | adapts slowly to real change |
| CUSUM / accumulated change | slow drift, many small steps | stop drifting (the attack ends) | delay; tuning of $k$, $h$ |
| robust statistics (MCD) | scattered contamination | concentrated, patient poison (notebook 08) | compute; breakdown at 50% |

**No defence is final; each changes the price.** Evaluate every detector against a stated threat model, with
constraints that keep the attack an attack, and report both the clean metrics and the robustness.